# Poker bot starter

**First:** choose **File → Save a copy in Drive**.

1. Edit and run the **Write your bot** cell.
2. Run **Check your bot**.
3. **Download & submit** your `.py` file through the host's invitation link.


## 1. Write your bot

Keep `%%writefile poker_bot.py`; re-run this cell after every edit. Only this cell becomes your submission.

**Rules:** one file, ≤20 KB; exactly `def act(state)`; ≤500 ms per decision. Use ordinary functions, loops, expressions, and top-level helpers. Imports: only approved members of `math`, `random`, `itertools`, `collections`.

No classes, annotations, decorators, nested functions, introspection, `eval`/`exec`, files, network, external packages, trained models, or imported poker engines. Compute your strategy; do not embed datasets/weights. Globals reset every decision. [Full rules](https://midnight-poker-circuit.vercel.app/examples/BOT_PROTOCOL.md).


In [ ]:
%%writefile poker_bot.py
"""Edit act(state), export this .py file, and upload it at the host's /submit page.
Only your hole cards and public information are supplied. Return one legal action.
"""


def act(state):
    # Replace this simple baseline with your own poker algorithm.
    ranks = "23456789TJQKA"
    first, second = state["holeCards"]
    pair = first[0] == second[0]
    high_cards = ranks.index(first[0]) + ranks.index(second[0]) >= 19

    if state["street"] == "preflop" and state["canRaise"] and (pair or high_cards):
        return {"action": "raise", "raiseTo": min(state["minRaiseTo"], state["maxRaiseTo"])}

    # A free check is always legal. Calling costs at most your remaining stack.
    if state["toCall"] == 0:
        return {"action": "check"}
    price = min(state["toCall"], state["stack"]) / max(1, state["pot"] + min(state["toCall"], state["stack"]))
    if not pair and not high_cards and price > 0.25:
        return {"action": "fold"}
    return {"action": "call"}

### What exactly is `state`?

A **Python dictionary**: the table snapshot immediately before **your turn**. The server calls `act(state)` and applies your returned action. You see your cards and public information—never opponents' cards, future cards, or the shuffle seed.

| Key | Meaning |
|---|---|
| `holeCards` | Your two cards, e.g. `["As", "Kd"]` |
| `board` | 0/3/4/5 revealed community cards |
| `street` | `preflop`, `flop`, `turn`, or `river` |
| `stack` / `bet` | Your uncommitted chips / chips committed this street |
| `pot` | Everyone's total contributions this hand, including current bets |
| `toCall` / `currentBet` | Chips needed to call / highest street contribution |
| `minRaiseTo` / `maxRaiseTo` | Minimum full-raise total / your all-in total (`bet + stack`) |
| `canRaise` / `legalActions` | Raising allowed? / available action names |
| `seat` / `dealer` | Your seat / dealer seat; zero-based indexes |
| `smallBlind` / `bigBlind` | Current blind amounts |
| `hand` / `schemaVersion` | Table hand number (starts at 1) / `1` |
| `players` | All seats, including you and eliminated players; fields below |
| `history` | Last ≤30 actions **from this hand**, oldest first |

**Cards:** rank `23456789TJQKA` + suit `c/d/h/s` (clubs/diamonds/hearts/spades).

**Each player:** `id`, `name`, `seat`, `stack`, `bet`, `contributed` (this hand), `folded`, `out` (eliminated), `allIn`.
**Each history item:** `seat`, `street`, `action`, `bet` (street total after acting). No previous hands or tournament scores/stage.

### Return one action

```python
{"action": "fold"}
{"action": "check"}                # toCall == 0
{"action": "call"}                 # pays min(toCall, stack)
{"action": "raise", "raiseTo": 120} # TOTAL street contribution
```

Raise only if `canRaise`: integer `minRaiseTo ≤ raiseTo ≤ maxRaiseTo`; a smaller all-in `maxRaiseTo` is also legal. Example: `bet=20`, `raiseTo=120` adds **100** chips.


## 2. Check your bot

Run after the bot cell. Four sample decisions check basic compatibility, **not strategy strength or full server approval**. These local checks run your code; they are not a sandbox.


In [ ]:
#@title Run checks (4 sample decisions)
from pathlib import Path
import ast
import copy
import json
import subprocess
import sys

source_path = Path("poker_bot.py")
source = source_path.read_text(encoding="utf-8")
assert len(source.encode("utf-8")) <= 20000, "The source must be at most 20 KB."
tree = ast.parse(source, filename=str(source_path))
entries = [node for node in tree.body if isinstance(node, ast.FunctionDef) and node.name == "act"]
assert len(entries) == 1, "Define exactly one top-level def act(state)."
args = entries[0].args
assert (len(args.args) == 1 and args.args[0].arg == "state" and not args.posonlyargs
        and not args.kwonlyargs and not args.vararg and not args.kwarg
        and not args.defaults and not args.kw_defaults), "Use exactly def act(state)."
approved_imports = {"math", "random", "itertools", "collections"}
for node in ast.walk(tree):
    if isinstance(node, ast.Import):
        assert all(item.name in approved_imports for item in node.names), "Only the four approved libraries may be imported."
    if isinstance(node, ast.ImportFrom):
        assert not node.level and node.module in approved_imports, "Only the four approved libraries may be imported."
        assert all(item.name != "*" for item in node.names), "Use explicit imports, not import *."

CASES = [{'name': 'Premium preflop', 'state': {'schemaVersion': 1, 'hand': 1, 'seat': 0, 'dealer': 4, 'smallBlind': 10, 'bigBlind': 20, 'stack': 980, 'bet': 20, 'pot': 120, 'toCall': 20, 'currentBet': 40, 'minRaiseTo': 60, 'maxRaiseTo': 1000, 'canRaise': True, 'legalActions': ['fold', 'call', 'raise'], 'history': [{'seat': 1, 'street': 'preflop', 'action': 'raise', 'bet': 40}], 'players': [{'id': 0, 'name': 'Opponent1', 'seat': 0, 'stack': 980, 'bet': 20, 'contributed': 20, 'folded': False, 'out': False, 'allIn': False}, {'id': 1, 'name': 'Opponent2', 'seat': 1, 'stack': 960, 'bet': 40, 'contributed': 40, 'folded': False, 'out': False, 'allIn': False}, {'id': 2, 'name': 'Opponent3', 'seat': 2, 'stack': 980, 'bet': 20, 'contributed': 20, 'folded': False, 'out': False, 'allIn': False}, {'id': 3, 'name': 'Opponent4', 'seat': 3, 'stack': 980, 'bet': 20, 'contributed': 20, 'folded': False, 'out': False, 'allIn': False}, {'id': 4, 'name': 'Opponent5', 'seat': 4, 'stack': 980, 'bet': 20, 'contributed': 20, 'folded': False, 'out': False, 'allIn': False}], 'street': 'preflop', 'holeCards': ['As', 'Ad'], 'board': []}}, {'name': 'Free flop check', 'state': {'schemaVersion': 1, 'hand': 1, 'seat': 0, 'dealer': 4, 'smallBlind': 10, 'bigBlind': 20, 'stack': 970, 'bet': 0, 'pot': 150, 'toCall': 0, 'currentBet': 0, 'minRaiseTo': 20, 'maxRaiseTo': 970, 'canRaise': True, 'legalActions': ['check', 'fold', 'raise'], 'history': [], 'players': [{'id': 0, 'name': 'Opponent1', 'seat': 0, 'stack': 970, 'bet': 0, 'contributed': 30, 'folded': False, 'out': False, 'allIn': False}, {'id': 1, 'name': 'Opponent2', 'seat': 1, 'stack': 970, 'bet': 0, 'contributed': 30, 'folded': False, 'out': False, 'allIn': False}, {'id': 2, 'name': 'Opponent3', 'seat': 2, 'stack': 970, 'bet': 0, 'contributed': 30, 'folded': False, 'out': False, 'allIn': False}, {'id': 3, 'name': 'Opponent4', 'seat': 3, 'stack': 970, 'bet': 0, 'contributed': 30, 'folded': False, 'out': False, 'allIn': False}, {'id': 4, 'name': 'Opponent5', 'seat': 4, 'stack': 970, 'bet': 0, 'contributed': 30, 'folded': False, 'out': False, 'allIn': False}], 'street': 'flop', 'holeCards': ['7c', '2d'], 'board': ['Kh', '9s', '4c']}}, {'name': 'River facing a bet', 'state': {'schemaVersion': 1, 'hand': 1, 'seat': 0, 'dealer': 4, 'smallBlind': 10, 'bigBlind': 20, 'stack': 900, 'bet': 0, 'pot': 700, 'toCall': 200, 'currentBet': 200, 'minRaiseTo': 400, 'maxRaiseTo': 900, 'canRaise': True, 'legalActions': ['fold', 'call', 'raise'], 'history': [{'seat': 1, 'street': 'preflop', 'action': 'raise', 'bet': 40}], 'players': [{'id': 0, 'name': 'Opponent1', 'seat': 0, 'stack': 900, 'bet': 0, 'contributed': 100, 'folded': False, 'out': False, 'allIn': False}, {'id': 1, 'name': 'Opponent2', 'seat': 1, 'stack': 700, 'bet': 200, 'contributed': 300, 'folded': False, 'out': False, 'allIn': False}, {'id': 2, 'name': 'Opponent3', 'seat': 2, 'stack': 900, 'bet': 0, 'contributed': 100, 'folded': False, 'out': False, 'allIn': False}, {'id': 3, 'name': 'Opponent4', 'seat': 3, 'stack': 900, 'bet': 0, 'contributed': 100, 'folded': False, 'out': False, 'allIn': False}, {'id': 4, 'name': 'Opponent5', 'seat': 4, 'stack': 900, 'bet': 0, 'contributed': 100, 'folded': False, 'out': False, 'allIn': False}], 'street': 'river', 'holeCards': ['Ah', 'Jh'], 'board': ['Kh', 'Th', '2c', '4d', '8h']}}, {'name': 'Short stack call', 'state': {'schemaVersion': 1, 'hand': 1, 'seat': 0, 'dealer': 4, 'smallBlind': 10, 'bigBlind': 20, 'stack': 30, 'bet': 20, 'pot': 180, 'toCall': 80, 'currentBet': 100, 'minRaiseTo': 180, 'maxRaiseTo': 50, 'canRaise': False, 'legalActions': ['fold', 'call'], 'history': [{'seat': 1, 'street': 'preflop', 'action': 'raise', 'bet': 40}], 'players': [{'id': 0, 'name': 'Opponent1', 'seat': 0, 'stack': 30, 'bet': 20, 'contributed': 20, 'folded': False, 'out': False, 'allIn': False}, {'id': 1, 'name': 'Opponent2', 'seat': 1, 'stack': 900, 'bet': 100, 'contributed': 100, 'folded': False, 'out': False, 'allIn': False}, {'id': 2, 'name': 'Opponent3', 'seat': 2, 'stack': 980, 'bet': 20, 'contributed': 20, 'folded': False, 'out': False, 'allIn': False}, {'id': 3, 'name': 'Opponent4', 'seat': 3, 'stack': 980, 'bet': 20, 'contributed': 20, 'folded': False, 'out': False, 'allIn': False}, {'id': 4, 'name': 'Opponent5', 'seat': 4, 'stack': 980, 'bet': 20, 'contributed': 20, 'folded': False, 'out': False, 'allIn': False}], 'street': 'preflop', 'holeCards': ['Qs', 'Qd'], 'board': []}}]

# This test harness is outside the saved bot file. Each sample uses a fresh process.
runner = r"""
import contextlib
import json
import sys
with open(sys.argv[1], encoding="utf-8") as file:
    source = file.read()
state = json.loads(sys.stdin.read())
namespace = {"__name__": "student_bot"}
with contextlib.redirect_stdout(sys.stderr):
    exec(compile(source, sys.argv[1], "exec"), namespace)
    action = namespace["act"](state)
print(json.dumps(action, allow_nan=False))
"""

def check_action(action, state):
    assert isinstance(action, dict), "Return a dictionary."
    assert action.get("action") in {"fold", "check", "call", "raise"}, "Unknown or missing action."
    expected = {"action", "raiseTo"} if action["action"] == "raise" else {"action"}
    assert set(action) == expected, "Use only action and, for a raise, raiseTo."
    if action["action"] == "check":
        assert state["toCall"] == 0, "Cannot check while facing a bet."
    if action["action"] == "raise":
        target = action["raiseTo"]
        assert state["canRaise"], "Raising is not currently allowed."
        assert type(target) is int, "raiseTo must be an integer."
        assert state["currentBet"] < target <= state["maxRaiseTo"], "Raise target is outside your available chips."
        assert target >= state["minRaiseTo"] or target == state["maxRaiseTo"], "Raise too small unless it is all-in."

failures = []
for case in CASES:
    try:
        completed = subprocess.run(
            [sys.executable, "-I", "-B", "-c", runner, str(source_path.resolve())],
            input=json.dumps(copy.deepcopy(case["state"])), capture_output=True,
            text=True, timeout=1, check=True,
        )
        action = json.loads(completed.stdout)
        check_action(action, case["state"])
        print("PASS", case["name"], "→", action)
    except (AssertionError, ValueError, subprocess.SubprocessError) as error:
        detail = (getattr(error, "stderr", None) or str(error)).strip()
        failures.append(case["name"])
        print("FAIL", case["name"], "→", detail)

assert not failures, "Fix these local checks before uploading: " + ", ".join(failures)
print("\n4/4 local smoke checks passed. Upload the .py file for authoritative server validation.")

## 3. Download & submit

Run below, then upload **`poker_bot.py`** through your host's invitation link. Choose a bot name, accept the rules, and wait for **Accepted**. Save your private receipt to update the same entry later.

The host starts the tournament; you can close Colab. Test/download helpers stay outside your bot file.


In [ ]:
from google.colab import files
files.download("poker_bot.py")